# PROD — Monitor & Detect Drift

**Duration:** 5 min

## CloudWatch Metrics

In [ ]:
import boto3, time
cw = boto3.client('cloudwatch', region_name='us-east-1')

def log_prediction(prob: float, latency_ms: float):
    cw.put_metric_data(
        Namespace='ChurnAPI',
        MetricData=[
            {'MetricName': 'PredictionProbability', 'Value': prob,        'Unit': 'None'},
            {'MetricName': 'Latency',               'Value': latency_ms,  'Unit': 'Milliseconds'},
            {'MetricName': 'RequestCount',          'Value': 1,           'Unit': 'Count'},
        ]
    )

# In your /predict endpoint:
@app.post('/predict')
def predict(customer: CustomerFeatures):
    start = time.time()
    # ... prediction logic ...
    log_prediction(prob, (time.time()-start)*1000)
    return result

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/mlops-real-world/mlops-prod-monitor.ipynb)


## Data Drift Detection with Evidently

In [ ]:
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset
import pandas as pd

# Reference = training data, Current = last 7 days of production requests
reference = pd.read_csv('data/telco_churn.csv').drop(columns=['customerID','Churn'])
current   = pd.read_csv('data/production_requests_last_7d.csv')

report = Report(metrics=[DataDriftPreset()])
report.run(reference_data=reference, current_data=current)
report.save_html('drift_report.html')

# Check if drift detected
result = report.as_dict()
drifted = result['metrics'][0]['result']['dataset_drift']
if drifted:
    print('⚠️  DATA DRIFT DETECTED — consider retraining')
    # Send SNS alert
    boto3.client('sns').publish(
        TopicArn='arn:aws:sns:us-east-1:<account>:ml-alerts',
        Message='Data drift detected in churn model. Review drift_report.html.',
        Subject='MLOps Alert: Data Drift'
    )

> **💡 Tip:** Schedule this drift check as a daily Lambda or cron job. When drift is detected, trigger a retraining pipeline automatically.

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is data drift in production ML?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4372738048" value="0">
      <span>Model weights changing over time</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4372738048" value="1">
      <span>The statistical distribution of input features shifting from what the model was trained on</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4372738048" value="2">
      <span>Prediction latency increasing</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q4372738048" value="3">
      <span>Database schema changes</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>